# Extraction pipeline

This notebook runs the full `citegraph` extraction interactively, from the folder of
PDFs to the finished data objects, and then queries the resulting graph to check that
the outputs are sane. It produces no manuscript figures: those live in the companion
notebook [figures.ipynb](figures.ipynb), which reads the artifacts
written here.

Each stage method writes its output to `OUT_DIR` and reads its input from disk if called with no arguments. The current works model uses one role-free `w-` ID for every bibliographic entity.

The pipeline layout under `OUT_DIR`:

```
out/
  markdown/                 # stage 1: docling output, one .md per PDF
  metadata/<paper>.json     # stage 2: per-PDF Gemini metadata
  references/<paper>.json   # stage 3: per-PDF Gemini references
  enrichment/               # stage 5: per-work enrichment cache
  sources.csv               # stage 2: pre-canonical source rows
  citations_raw.csv         # stage 3: raw citation occurrences
  works.csv                 # stage 4: canonical works from sources + citations
  citation_graph.csv        # stage 4: work-to-work edges
  enriched_works.csv        # stage 5: optional enrichment
  authors.csv               # stage 6: canonical authors
  author_citations.csv      # stage 6: author-work occurrences
```

In `works.csv`, `ring == 0` identifies the canonical PDF-backed core and `source_file` records the processed full text. Re-running a stage reuses its caches.

## Local development setup

Run this section first. It installs the repository in editable mode into the active notebook kernel, so imports resolve to the code under `src/citegraph` rather than to a separately installed release. It also works whether Jupyter starts in the repository root or in `examples/`.

`PDF_DIR` defaults to `<repo>/pdfs`. To use PDFs elsewhere, set the `CITEGRAPH_PDF_DIR` environment variable or edit that assignment. If no PDF directory is available, the Stage 1 cell can reuse markdown already cached under `OUT_DIR`.

In [ ]:
from pathlib import Path

import citegraph
import pandas as pd
from citegraph import Pipeline

# The replication root is the folder holding citegraph_out/ and institutional/.
# The library reads GOOGLE_API_KEY / OPENALEX_API_KEY from the environment or a
# .env file in the working directory; only stages that miss their cache need them.
ROOT = next(
    p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "citegraph_out").is_dir() and (p / "institutional").is_dir()
)
print(f"citegraph {citegraph.__version__} from {Path(citegraph.__file__).parent}")

# The source PDFs are not distributed (see README); place them under data/.
PDF_DIR = ROOT / "data"
OUT_DIR = ROOT / "citegraph_out"

pipe = Pipeline(
    pdf_dir=PDF_DIR,
    out_dir=OUT_DIR,
    model="gemini-3.1-flash-lite",
    enrich=False,
    recursive=True,
    show_progress=True,
    ocr="auto",
    llm_concurrency=4,
)

print(f"PDF input:       {PDF_DIR}")
print(f"Pipeline output: {OUT_DIR}")

### Stage 1 — PDFs ➜ markdown

`docling` is slow on first run; cached markdown is reused on subsequent calls. Pass `overwrite_markdown=True` on the `Pipeline` constructor if you need to force a re-conversion.

In [ ]:
pdf_paths = sorted(PDF_DIR.rglob("*.pdf")) if PDF_DIR.is_dir() else []

if pdf_paths:
    markdown_paths = pipe.convert_pdfs()
else:
    markdown_paths = sorted(pipe.layout.markdown_dir.glob("*.md"))
    if not markdown_paths:
        raise FileNotFoundError(
            f"No PDFs found under {PDF_DIR} and no cached markdown found under "
            f"{pipe.layout.markdown_dir}. Set PDF_DIR (or CITEGRAPH_PDF_DIR) to your PDF folder."
        )
    print(f"No PDFs found under {PDF_DIR}; using cached markdown.")

print(f"{len(markdown_paths)} markdown files in {pipe.layout.markdown_dir}")
for p in markdown_paths[:5]:
    print(" ", p.name)

In [ ]:
# # Spot-check the first markdown file before continuing.
# if markdown_paths:
#     print(markdown_paths[0].read_text()[:1500])

### Stage 2 — markdown ➜ source-work metadata

Calls Gemini once per PDF. Per-PDF JSON is cached under `OUT_DIR/metadata/`; rerunning rewrites `sources.csv`. These rows are pre-canonical, so duplicate PDFs are resolved in Stage 4.

In [ ]:
sources = pipe.extract_paper_metadata()
sources.head()

In [ ]:
sources.shape

### Stage 3 — markdown ➜ raw citation occurrences

Also Gemini-driven and cached per PDF under `OUT_DIR/references/`. Output is `citations_raw.csv`, with one row per raw citation occurrence and a role-free `citing_id`.

In [ ]:
citations_raw = pipe.extract_paper_references()
print(f"{len(citations_raw)} raw citations across {citations_raw['citing_id'].nunique()} core works")
citations_raw.head()

In [ ]:
citations_raw

### Stage 4 — canonical works + citation graph

Pure Python; no network. Sources and raw citations are canonicalized together, allowing duplicate PDFs to merge and citations to resolve to core works. Pass a custom `DedupConfig` to `Pipeline(..., dedup_config=...)` when tuning the matching threshold, weights, or year window.

#### Preview merged citation forms before writing Stage 4 outputs

`canonicalize_works` returns a `citation_cluster_ids` list aligned with `citations_raw`. This makes raw citation forms that resolved to the same canonical work auditable without writing stage outputs.

In [ ]:
from citegraph.dedup import canonicalize_works

preview_works, preview_graph, preview_stats = canonicalize_works(
    sources,
    citations_raw,
    pipe.dedup_config,
    show_progress=True,
)

clustered_citations = citations_raw.copy()
clustered_citations["canonical_id"] = preview_stats["citation_cluster_ids"]
merged_groups = (
    clustered_citations
    .groupby("canonical_id")
    .filter(lambda group: len(group) > 1)
    .sort_values(["canonical_id", "Title"])
)

print(
    f"{len(sources)} sources + {len(citations_raw)} raw citations -> "
    f"{len(preview_works)} canonical works and {len(preview_graph)} edges; "
    f"{merged_groups['canonical_id'].nunique()} merged citation clusters"
)

merged_groups[["canonical_id", "Title", "Authors", "Year", "Journal", "citing_id"]]

In [ ]:
merged_summary = (
    merged_groups
    .groupby("canonical_id")
    .agg(
        n_raw_citations=("Title", "size"),
        titles=("Title", list),
        authors=("Authors", list),
        years=("Year", list),
        citing_works=("citing_id", list),
    )
    .sort_values("n_raw_citations", ascending=False)
)

merged_summary

In [ ]:
# helper function to print a merged group in a readable format
def print_row(record):
    for col, value in record.items():
        print(f"\n{col}:")
        if isinstance(value, list):
            for i, item in enumerate(value, start=1):
                print(f"  {i}. {item}")
        else:
            print(value)
            
# Check one example
row = 0
print_row(merged_summary.iloc[row])

In [ ]:
works, graph = pipe.deduplicate()
print(
    f"{len(sources)} sources + {len(citations_raw)} raw citations -> "
    f"{len(works)} canonical works, {len(graph)} edges"
)
works.head()

In [ ]:
works

In [ ]:
graph

### Stage 5 — CrossRef / OpenAlex enrichment

Enrichment is optional but important for author identity confidence. It tries CrossRef first, then OpenAlex over every canonical work, including ring 0, and writes `enriched_works.csv` plus per-work cache files under `OUT_DIR/enrichment/`.

The key author-normalization column is `OpenAlex_Authors`: Stage 6 treats OpenAlex author IDs and ORCIDs as stronger identity evidence than string matching.

Use a real contact email and modest `max_workers`; providers can rate-limit aggressive batches, while re-runs reuse the cache. Review `enrichment_summary.json` and `enrichment_misses.csv` after the run.

In [ ]:
from citegraph.enrich import EnrichConfig

ENRICH_CONTACT_EMAIL = ""  # real runs should identify you to provider polite pools

enrich_config = EnrichConfig(
    contact_email=ENRICH_CONTACT_EMAIL,
    title_match_threshold=90.0,
    timeout_s=15.0,
    max_workers=2,
)

print(f"works.csv exists: {pipe.layout.works_csv.exists()}")
print(f"enriched_works.csv exists: {pipe.layout.enriched_works_csv.exists()}")
print(f"enrichment cache dir: {pipe.layout.enrichment_dir}")

In [ ]:
# Network call: run only when ready to query CrossRef/OpenAlex.
# Re-runs reuse per-work cache files in OUT_DIR/enrichment/.
pipe_enriched = Pipeline(
    pdf_dir=PDF_DIR,
    out_dir=OUT_DIR,
    model="gemini-3.1-flash-lite",
    enrich=True,
    enrich_config=enrich_config,
    recursive=True,
    show_progress=True,
    ocr="auto",
    llm_concurrency=4,
)

enriched_works = pipe_enriched.maybe_enrich()
enriched_works.head()

In [ ]:
# Coverage summary: how much enrichment did we get?
import json

enriched_works = pd.read_csv(pipe.layout.enriched_works_csv, index_col="id")

n_works = len(enriched_works)
status_counts = enriched_works.get("enrichment_status", pd.Series(dtype=str)).value_counts(dropna=False)
source_counts = enriched_works.get("enrichment_source", pd.Series(dtype=str)).fillna("miss").value_counts(dropna=False)

if pipe.layout.enrichment_summary_json.exists():
    enrichment_summary = json.loads(pipe.layout.enrichment_summary_json.read_text())
    print(json.dumps(enrichment_summary, indent=2))
else:
    n_matched = int((enriched_works.get("enrichment_status") == "matched").sum()) if "enrichment_status" in enriched_works else 0
    print(f"Matched {n_matched}/{n_works} works ({n_matched / n_works:.1%}).")

print()
print("Status counts:")
print(status_counts)
print()
print("Provider counts:")
print(source_counts)

enriched_works[[
    column for column in [
        "ring", "Title", "Authors", "Year", "Journal", "doi", "enrichment_source",
        "enrichment_status", "enrichment_title_score", "enrichment_year_match",
        "OpenAlex_Authors",
    ]
    if column in enriched_works.columns
]].head(10)

In [ ]:
# Inspect works that CrossRef/OpenAlex could not match confidently.
if pipe.layout.enrichment_misses_csv.exists():
    enrichment_misses = pd.read_csv(pipe.layout.enrichment_misses_csv)
    print(f"{len(enrichment_misses)} enrichment misses")
    display_cols = [
        column for column in ["id", "ring", "Title", "Authors", "Year", "Journal", "enrichment_miss_reason"]
        if column in enrichment_misses.columns
    ]
    display(enrichment_misses[display_cols].head(25))
else:
    print("No enrichment_misses.csv file found.")

In [ ]:
# Inspect the weakest accepted work matches first.
if {"enrichment_status", "enrichment_title_score"}.issubset(enriched_works.columns):
    weak_matches = (
        enriched_works[enriched_works["enrichment_status"] == "matched"]
        .sort_values("enrichment_title_score")
    )
    display_cols = [
        column for column in [
            "ring", "Title", "Authors", "Year", "doi", "enrichment_source",
            "enrichment_title_score", "enrichment_candidate_title", "enrichment_year_match",
        ]
        if column in weak_matches.columns
    ]
    display(weak_matches[display_cols].head(15))
else:
    print("Diagnostic enrichment columns are missing; rerun Stage 5 with current code.")

In [ ]:
# Check whether enriched works carry structured OpenAlex author identity evidence.
from citegraph.io import parse_openalex_authors

if "OpenAlex_Authors" in enriched_works.columns:
    openalex_author_lists = enriched_works["OpenAlex_Authors"].map(parse_openalex_authors)
    rows_with_author_ids = openalex_author_lists.map(
        lambda authors: any(
            author.get("openalex_id") or author.get("orcid")
            for author in (authors or [])
            if isinstance(author, dict)
        )
    )
    print(
        f"Works with OpenAlex/ORCID author evidence: "
        f"{rows_with_author_ids.sum()}/{len(rows_with_author_ids)} "
        f"({rows_with_author_ids.mean():.1%})"
    )
else:
    print("OpenAlex_Authors is missing; author normalization will use string-only evidence.")

After enrichment, rerun Stage 6 below. `pipe.normalize_authors()` automatically reads `enriched_works.csv` and uses `OpenAlex_Authors` as identity evidence.

### Stage 6 — author normalization

Clusters every author across canonical works into canonical records and writes `authors.csv` plus `author_citations.csv`.

The default `merge_mode="strict"` is precision-first. Pass `author_config=AuthorClusterConfig(merge_mode="loose")` to collapse by `(surname, first_initial)`. `author_review.json` flags initial-only ambiguity, possible uncorroborated compound surnames, and corporate authors. To force a merge, add `cluster_id,canonical_id` rows to `author_aliases.csv` and rerun this local stage.

In [ ]:
authors_df, citations_df = pipe.normalize_authors()

In [ ]:
print(
    f"{len(citations_df)} author occurrences -> "
    f"{len(authors_df)} canonical authors"
)
authors_df.head(15)

#### Preview which raw author names were joined together

`authors_df` has one row per canonical author. `citations_df` has every raw author occurrence that was assigned to those canonical authors. The cells below show the canonical authors that absorbed multiple raw name forms.

In [ ]:
author_occurrences = citations_df.merge(
    authors_df[["display_name"]],
    left_on="author_id",
    right_index=True,
    how="left",
)

author_join_summary = (
    author_occurrences
    .groupby("author_id")
    .agg(
        canonical_name=("display_name", "first"),
        n_occurrences=("raw_author", "size"),
        n_distinct_works=("record_id", "nunique"),
        raw_author_forms=("raw_author", lambda values: sorted(set(values))),
        work_ids=("record_id", lambda values: sorted(set(values))),
    )
)
author_join_summary["n_raw_forms"] = author_join_summary["raw_author_forms"].map(len)

author_joins = (
    author_join_summary
    .query("n_raw_forms > 1")
    .sort_values(
        ["n_occurrences", "n_raw_forms", "canonical_name"],
        ascending=[False, False, True],
    )
)

print(f"{len(author_joins)} canonical authors merged multiple raw name forms")
author_joins[["canonical_name", "n_occurrences", "n_distinct_works", "n_raw_forms", "raw_author_forms", "work_ids"]]

In [ ]:
def print_author_join(row_or_author_id=0):
    if isinstance(row_or_author_id, int):
        author_id = author_joins.index[row_or_author_id]
        record = author_joins.iloc[row_or_author_id]
    else:
        author_id = row_or_author_id
        record = author_joins.loc[author_id]

    print(f"author_id: {author_id}")
    for column, value in record.items():
        print()
        print(f"{column}:")
        if isinstance(value, list):
            for index, item in enumerate(value, start=1):
                print(f"  {index}. {item}")
        else:
            print(value)

    return author_occurrences.loc[
        author_occurrences["author_id"] == author_id,
        ["raw_author", "record_id", "position"],
    ].sort_values(["raw_author", "record_id"])


# Change 0 to another row number, or pass an author id.
print_author_join(3)

## Analysis — query the finished citation graph

At this point the pipeline has written the main artifacts to disk. The cleanest analysis path is to load one `CitationGraph` view and use it for common questions. Keep the raw DataFrames nearby only when you want custom pandas analysis.

In [ ]:
from citegraph import CitationGraph
import pandas as pd

# Load from disk so these cells work whether you ran the pipeline all at once or progressively.
g = CitationGraph.from_out_dir(OUT_DIR)
print(g)
print(
    f"core_works={g.n_core_works}, "
    f"works={g.n_works}, "
    f"citation_edges={g.n_edges}, "
    f"authors={len(g.authors) if g.has_authors else 0}"
)

# Optional aliases for direct DataFrame work.
core = g.core
works = g.works
graph = g.edges
authors = g.authors
author_citations = g.author_citations


def print_record(record):
    """Print a Series-like row with list values expanded line by line."""
    for column, value in record.items():
        print()
        print(f"{column}:")
        if isinstance(value, list):
            for index, item in enumerate(value, start=1):
                print(f"  {index}. {item}")
        else:
            print(value)

### Top-cited works

`g.top_cited()` counts distinct citing works for each cited canonical work. Cited core works can rank alongside works discovered from bibliographies.

In [ ]:
top_works = g.top_cited(10)
top_works

In [ ]:
# Print one top-cited work with all fields visible.
row = 0
print_record(top_works.iloc[row])

### Cited-work drill-down

Pick a cited work and ask which canonical works cite it. The citing side can include any processed core work, while the cited work may itself belong to the core.

In [ ]:
top_work_id = top_works.index[0]
print(f"Works that cite:\n  {top_work_id}\n")
g.citers_of(top_work_id)

### Core-work drill-down

Pick one core work and list the canonical works it cites. This checks whether its bibliography was extracted and canonicalized sensibly.

In [ ]:
sample_core_id = g.core.index[0]
print(f"Works cited by:\n  {sample_core_id}\n")
g.cited_by(sample_core_id)

## Author analysis

These queries use canonical authors produced by Stage 6. Avoid splitting `works["Authors"]` by commas; use `Authors_List` or the normalized author tables.

The cells below ask which core works cited works by a target author and which core-work journals did the citing.

In [ ]:
if not g.has_authors:
    raise RuntimeError("Author tables are not loaded. Run pipe.normalize_authors() first.")

top_authors = g.top_cited_authors(15)
top_authors

In [ ]:
# Diacritic-insensitive substring lookup on surname or display name.
# Change this query to inspect a different author.
target_author_query = "cardenas"

author_matches = g.find_author(target_author_query)
author_matches

In [ ]:
# Pick the top match. If the result list shows split identities, inspect those first.
target_author_id = author_matches.index[0]
target_author_name = author_matches.loc[target_author_id, "display_name"]

print(f"Selected author:\n  {target_author_id} — {target_author_name}\n")

# One row per source-paper -> cited-reference edge for works by this author.
author_context = g.citation_context_for_author(target_author_id)
author_context

In [ ]:
# Distinct citing works. A work that cites several works by this author counts once.
works_citing_author = g.citing_papers_by_author(target_author_id)

print(
    f"{len(works_citing_author)} distinct works cite at least one "
    f"work by {target_author_name}."
)
works_citing_author

In [ ]:
# Journal distribution on the citing side of the graph.
citing_journals = g.source_journals_citing_author(target_author_id)
citing_journals

## Optional NetworkX metrics

`to_networkx()` exports a directed work-to-work graph. Edges go from citing work to cited work, nodes carry `ring` and `source_file`, and core-to-core edges are possible.

In [ ]:
import networkx as nx

G = g.to_networkx()
metrics = pd.DataFrame({
    "ring": pd.Series(nx.get_node_attributes(G, "ring")),
    "source_file": pd.Series(nx.get_node_attributes(G, "source_file")),
    "in_degree": pd.Series(dict(G.in_degree())),
    "out_degree": pd.Series(dict(G.out_degree())),
    "pagerank": pd.Series(nx.pagerank(G, alpha=0.85)),
})
metrics.sort_values("pagerank", ascending=False).head(15)

---

## Next: manuscript figures

Every data object the paper needs now exists under `OUT_DIR`. Open
[figures.ipynb](figures.ipynb) to build the manuscript figures from
them. That notebook reads these artifacts only and never calls the extraction
model, so it can be rerun freely.